# Tools

Generated from the book sources. Do not edit by hand: changes belong in the `.qmd` chapter.

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
import requests

load_dotenv()

client = OpenAI()

CHAT_MODEL = os.environ["CHAT_MODEL"]

### Tool schema for the `add_numbers` function

`lst-add-numbers-tool-schema`

In [ ]:
add_numbers_tool = {
  "type": "function",
  "function": {
    "name": "add_numbers",
    "description": "Add two numbers and return the sum.",
    "parameters": {
      "type": "object",
      "properties": {
        "a": {"type":"number", "description":"First addend"},
        "b": {"type":"number", "description":"Second addend"}
      },
      "required": ["a","b"],
      "additionalProperties": False
    }
  }
}

In [ ]:
def add_numbers(a: float, b: float):
    return {"result": a + b}

In [ ]:
messages = [
  {
    "role": "system",
    "content": (
      "You may call tools if needed. "
      "Show the final result clearly."
    ),
  },
  {
    "role": "user",
    "content": "What is 13 plus 29? Use the tool if helpful.",
  },
]

resp = client.chat.completions.create(
  model=CHAT_MODEL,
  messages=messages,
  tools=[add_numbers_tool],
  tool_choice="auto",
  temperature=0.2,
  seed=42
)

print("finish_reason:", resp.choices[0].finish_reason)
print(json.dumps(
    resp.choices[0].message.model_dump(), indent=2
))

In [ ]:
msg = resp.choices[0].message
if msg.tool_calls:
    for tc in msg.tool_calls:
        if tc.function.name == "add_numbers":
            args = json.loads(tc.function.arguments)
            result = add_numbers(**args)
            print(json.dumps(result))
            # Append assistant tool call + tool result
            messages.append({
                "role": "assistant",
                "tool_calls": [tc],
            })
            messages.append({
              "role": "tool",
              "tool_call_id": tc.id,
              "content": json.dumps(result),
            })

In [ ]:
final = client.chat.completions.create(
  model=CHAT_MODEL,
  messages=messages,
  temperature=0.2,
  seed=42
)

print(final.choices[0].message.content)

In [ ]:
geo_raw = requests.get(
    "https://geocoding-api.open-meteo.com/v1/search?name=Berlin&count=1",
    timeout=10,
).json()

lines = json.dumps(geo_raw, indent=2).splitlines()
print("\n".join(lines[:14]))
print(f"  ... ({len(lines) - 14} more lines)")

In [ ]:
lat = geo_raw["results"][0]["latitude"]
lon = geo_raw["results"][0]["longitude"]
forecast_raw = requests.get(
    "https://api.open-meteo.com/v1/forecast"
    f"?latitude={lat}&longitude={lon}&current_weather=true",
    timeout=10,
).json()
print(json.dumps(forecast_raw, indent=2))

### The `get_weather` helper function, reused throughout this chapter

`lst-get-weather-function`

In [ ]:
def get_weather(city: str, unit: str = "C"):
    geo = requests.get(
        f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1",
        timeout=10,
    ).json()
    if not geo.get("results"):
        return {"error": "City not found"}

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}&current_weather=true"
    )
    r = requests.get(url, timeout=10).json()
    temp_c = r["current_weather"]["temperature"]

    if unit == "C":
        return {"temperature": temp_c, "unit": "C"}
    else:
        return {"temperature": temp_c * 9 / 5 + 32, "unit": "F"}

### Tool schema for the `get_weather` function

`lst-weather-tool-schema`

In [ ]:
weather_tool = {
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get current weather for a city.",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "City name, e.g., Berlin"
        },
        "unit": {
          "type": "string",
          "enum": ["C", "F"],
          "default": "C",
          "description": "Temperature unit: C (Celsius) or F (Fahrenheit)."
        }
      },
      "required": ["city"],
      "additionalProperties": False
    }
  }
}

In [ ]:
messages = [
  {"role": "system", "content": "You may call tools if needed."},
  {
    "role": "user",
    "content": (
      "What is the current temperature in Berlin "
      "in degrees Celsius?"
    ),
  },
]

resp = client.chat.completions.create(
  model=CHAT_MODEL,
  messages=messages,
  tools=[add_numbers_tool, weather_tool],
  tool_choice="auto",
  temperature=0.0,
  seed=42,
)

msg = resp.choices[0].message
print(json.dumps(msg.model_dump(), indent=2))

In [ ]:
import json as _json

if msg.tool_calls:
    # Record the assistant tool call
    messages.append({
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in msg.tool_calls
        ],
    })

    # Execute tools and add their results
    for tc in msg.tool_calls:
        args = _json.loads(tc.function.arguments or "{}")
        if tc.function.name == "get_weather":
            tool_result = get_weather(**args)
            print(_json.dumps(tool_result))
        elif tc.function.name == "add_numbers":
            tool_result = add_numbers(**args)
        else:
            tool_result = {"error": f"Unknown tool {tc.function.name}"}

        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": _json.dumps(tool_result),
        })

In [ ]:
final = client.chat.completions.create(
  model=CHAT_MODEL,
  messages=messages,
  seed=42,
)

print(final.choices[0].message.content)

### The `web_search` helper function wrapping the Serper.dev API

`lst-web-search-function`

In [ ]:
SERPER_API_KEY = os.getenv("SERPER_API_KEY")


def web_search(query: str, site: str | None = None, top_k: int = 5):
    """
    Call Serper.dev to obtain web search results.
    Optionally restricts results to a specific site.
    Return a compact JSON object with a list of
    title/link/snippet triples.
    """
    if SERPER_API_KEY is None:
        return {"error": "SERPER_API_KEY not set"}

    q = f"site:{site} {query}" if site else query
    payload = {"q": q, "num": max(1, min(top_k, 10))}
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json",
    }

    resp = requests.post(
        "https://google.serper.dev/search",
        headers=headers,
        data=json.dumps(payload),
        timeout=15,
    )
    data = resp.json()

    results = []
    for item in (data.get("organic") or [])[:top_k]:
        results.append({
            "title": item.get("title", ""),
            "link": item.get("link", ""),
            "snippet": item.get("snippet", ""),
        })

    return {"results": results, "source": "serper"}

### Tool schema for the `web_search` function

`lst-search-tool-schema`

In [ ]:
search_tool = {
  "type": "function",
  "function": {
    "name": "web_search",
    "description": (
      "Search the web and return titles, URLs, and "
      "snippets (via Serper.dev)."
    ),
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": (
            "A precise search query. You can include "
            "operators."
          )
        },
        "site": {
          "type": "string",
          "description": (
            "Optional domain restriction, e.g. "
            "'example.com'."
          )
        },
        "top_k": {
          "type": "integer",
          "description": "Maximum number of results to return (1–10).",
          "default": 5
        }
      },
      "required": ["query"],
      "additionalProperties": False
    }
  }
}

In [ ]:
user_task = (
  "I am considering a career transition into "
  "data analytics. Summarize typical "
  "responsibilities, core skills, and common "
  "tools for a data analyst role. Use web "
  "search if needed, prioritize reputable "
  "sources, and include the URLs you used."
)

messages = [
  {
    "role": "system",
    "content": (
      "You may call tools if needed. After using "
      "tools, cite the URLs you relied on."
    ),
  },
  {"role": "user", "content": user_task},
]

resp = client.chat.completions.create(
  model=CHAT_MODEL,
  messages=messages,
  tools=[search_tool],
  tool_choice="auto",
  temperature=0.0,
  seed=42,
)

msg = resp.choices[0].message
print(json.dumps(msg.model_dump(), indent=2))

In [ ]:
if msg.tool_calls:
    messages.append({
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in msg.tool_calls
        ],
    })

    tool_results = []
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments or "{}")
        if tc.function.name == "web_search":
            tool_result = web_search(**args)
        else:
            tool_result = {"error": f"Unknown tool {tc.function.name}"}

        tool_results.append(tool_result)
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tool_result),
        })

    preview = dict(tool_results[0])
    if "results" in preview:
        preview["results"] = preview["results"][:2]
    print(json.dumps(preview, indent=2))

In [ ]:
final = client.chat.completions.create(
  model=CHAT_MODEL,
  messages=messages,
  temperature=0.2,
  seed=42,
)

print(final.choices[0].message.content)

### A minimal MCP server wrapping the `get_weather` function

`lst-mcp-weather-server`

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("weather_tools")

@mcp.tool(name="get_weather")
async def get_weather_tool(city: str, unit: str = "C"):
    """Get current weather for a city."""
    return get_weather(city, unit)

### Discovering and calling the weather server's tool from an in-process MCP client

`lst-mcp-weather-client`

In [ ]:
from fastmcp import Client

async with Client(mcp) as mcp_client:
    tools = await mcp_client.list_tools()
    print([t.name for t in tools])

    result = await mcp_client.call_tool("get_weather", {"city": "Berlin"})
    print(result.data)